In [3]:
# ! pip install pandas numpy matplotlib seaborn scikit-learn

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Задание 1: Пропуски и фильтрация

Напиши код, который выполнит следующие шаги:

## Шаг А
- Заполни пропущенные значения (`NaN`) в столбце `age` средним возрастом по всей таблице.

## Шаг Б
- Отфильтруй таблицу: оставь только пользователей, у которых `purchase_amount` > 1000 и `city` != `Казань`.

## Шаг В
- Сгруппируй отфильтрованный датасет по `is_premium` и посчитай медиану `purchase_amount`.

In [5]:
# Наш «сырой» датасет
raw_data = {
    'user_id': [101, 102, 103, 104, 105, 106, 107],
    'city': ['Москва', 'Санкт-Петербург', 'Москва', 'Казань', 'Санкт-Петербург', 'Москва', 'Казань'],
    'age': [24, np.nan, 31, 45, 22, np.nan, 38], # np.nan — это пропуск в данных
    'purchase_amount': [2500, 12000, 500, 0, 4300, 15000, 1100],
    'is_premium': [False, True, False, False, True, True, False]
}

df = pd.DataFrame(raw_data)
print(df)

   user_id             city   age  purchase_amount  is_premium
0      101           Москва  24.0             2500       False
1      102  Санкт-Петербург   NaN            12000        True
2      103           Москва  31.0              500       False
3      104           Казань  45.0                0       False
4      105  Санкт-Петербург  22.0             4300        True
5      106           Москва   NaN            15000        True
6      107           Казань  38.0             1100       False


In [6]:
print(df.age.mean())

32.0


In [7]:
df.age = df['age'].fillna(df['age'].mean())  # Заполняем пропуски средним возрастом

df_filtered = df[(df['purchase_amount'] > 1000) & (df['city'] != 'Казань')]  # Фильтруем пользователей с покупками больше 1000 и не из Казани

print(df_filtered.groupby('is_premium')['purchase_amount'].median())  # Группируем по признаку "премиум" и считаем медиану суммы покупок

is_premium
False     2500.0
True     12000.0
Name: purchase_amount, dtype: float64


# Задание 2: Работа с датами

В реальных логах почти всегда есть даты. Нам нужно научиться правильно с ними работать, так как во временных рядах и аналитике это 50% успеха.

Представь, что к нашему датасету добавился столбец с датой регистрации пользователя, но он пришел в виде обычного текста.

## Шаг А
- Переведи `reg_date` из типа `object` (строка) в формат `datetime` с помощью `pd.to_datetime`.

## Шаг Б
- Создай новый столбец `reg_year`, в который запиши год регистрации.

## Шаг В
- Отфильтруй датасет, оставив только пользователей, зарегистрированных в 2026 году.

In [8]:
data_v2 = {
    'user_id': [101, 102, 103, 104, 105],
    'reg_date': ['2025-01-15', '2025-03-20', '2026-02-10', '2026-05-01', '2024-12-01'],
    'purchase_amount': [2500, 12000, 500, 3000, 4300]
}

df2 = pd.DataFrame(data_v2)

In [9]:
print(df2)

   user_id    reg_date  purchase_amount
0      101  2025-01-15             2500
1      102  2025-03-20            12000
2      103  2026-02-10              500
3      104  2026-05-01             3000
4      105  2024-12-01             4300


In [10]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   user_id          5 non-null      int64
 1   reg_date         5 non-null      str  
 2   purchase_amount  5 non-null      int64
dtypes: int64(2), str(1)
memory usage: 252.0 bytes


In [11]:
df2['reg_year'] = pd.to_datetime(df2['reg_date']).dt.year  # Извлекаем год из даты регистрации
df2_filtered = df2[(df2['reg_year'] == 2026)]
print(df2_filtered)

   user_id    reg_date  purchase_amount  reg_year
2      103  2026-02-10              500      2026
3      104  2026-05-01             3000      2026


# Неделя 3: Объединение данных (merge / JOIN)

В реальной жизни данные часто хранятся в отдельных таблицах: информация о пользователях — в одной, а транзакции — в другой. Чтобы анализировать данные, нужно уметь их правильно объединять.

В Pandas используется `merge()`, аналог `JOIN` в SQL.

## Задача
У тебя есть две таблицы:

- `users` — справочник клиентов
- `transactions` — история покупок

Обрати внимание:

- некоторые пользователи не делали покупок;
- некоторые транзакции принадлежат `user_id`, которого нет в `users`.

Ниже — исходные данные для практики.

In [12]:
# Таблица А: Клиенты
users = pd.DataFrame({
    'user_id': [101, 102, 103, 104],
    'name': ['Алексей', 'Мария', 'Иван', 'Елена'],
    'city': ['Москва', 'Сочи', 'Казань', 'Красноярск']
})

# Таблица Б: Транзакции (покупок может быть несколько на одного user_id)
transactions = pd.DataFrame({
    'transaction_id': [1, 2, 3, 4, 5],
    'user_id': [101, 102, 101, 105, 102], # Игнат (105) купил, но его нет в users
    'item': ['Смартфон', 'Ноутбук', 'Наушники', 'Книга', 'Чехол'],
    'price': [30000, 70000, 5000, 1000, 1500]
})

# Задание: Слияния и агрегация

Напиши код, который выполнит шаги:

## Шаг A — Inner Join
- Объедини `users` и `transactions`, чтобы остались только строки с `user_id`, присутствующим в обеих таблицах.
- Выведи результат.

## Шаг B — Left Join
- Объедини таблицы так, чтобы сохранились все клиенты из `users` даже при отсутствии транзакций.
- В результирующей таблице для отсутствующих транзакций появятся `NaN`.

## Шаг C — Агрегация
- Используя результат левого соединения, посчитай суммарную `price` по каждому `city`.
- Учти, что `NaN` при суммировании игнорируются.

In [13]:
print(users)
print(transactions)

   user_id     name        city
0      101  Алексей      Москва
1      102    Мария        Сочи
2      103     Иван      Казань
3      104    Елена  Красноярск
   transaction_id  user_id      item  price
0               1      101  Смартфон  30000
1               2      102   Ноутбук  70000
2               3      101  Наушники   5000
3               4      105     Книга   1000
4               5      102     Чехол   1500


In [14]:
inner_join = pd.merge(users, transactions, on='user_id', how='inner')  # Внутреннее соединение
print(inner_join)
print()
left_join = pd.merge(users, transactions, on='user_id', how='left')  # Левое соединение
print(left_join)
print()
print(left_join.groupby('city')['price'].sum())  # Суммируем покупки по городам


   user_id     name    city  transaction_id      item  price
0      101  Алексей  Москва               1  Смартфон  30000
1      101  Алексей  Москва               3  Наушники   5000
2      102    Мария    Сочи               2   Ноутбук  70000
3      102    Мария    Сочи               5     Чехол   1500

   user_id     name        city  transaction_id      item    price
0      101  Алексей      Москва             1.0  Смартфон  30000.0
1      101  Алексей      Москва             3.0  Наушники   5000.0
2      102    Мария        Сочи             2.0   Ноутбук  70000.0
3      102    Мария        Сочи             5.0     Чехол   1500.0
4      103     Иван      Казань             NaN       NaN      NaN
5      104    Елена  Красноярск             NaN       NaN      NaN

city
Казань            0.0
Красноярск        0.0
Москва        35000.0
Сочи          71500.0
Name: price, dtype: float64


# Модуль 2: EDA и статистика

Ты отлично справляешься с базовыми манипуляциями данными. Пора перейти на следующий уровень: исследовать данные перед тем, как передавать их алгоритмам машинного обучения.

Этот этап называется EDA (Exploratory Data Analysis). Здесь мы ищем аномалии, выбросы и скрытые зависимости.

## Задача: Борьба с выбросами
В Data Science «выбросы» (`outliers`) — это аномальные значения, которые могут сильно исказить модель.

У тебя есть данные по возрасту кандидатов. Некоторые значения явно ошибочные: например, `190` и `5`.

Скопируй исходный датасет:

In [15]:
df_candidates = pd.DataFrame({
    'candidate_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'age': [22, 25, 190, 30, 28, 5, 35, 42, 29, 31] 
    # Обрати внимание на возраст 190 и 5 — это явные аномалии (выбросы)
})

# Задание: Анализ выбросов

Напиши код, который выполнит шаги:

## Шаг A
- Посчитай среднее значение столбца `age`.

## Шаг B
- Посчитай медиану столбца `age`.
- В комментарии укажи, какой показатель (`среднее` или `медиана`) более устойчив к экстремальным выбросам (например, `190` и `5`).

## Шаг C
- Отфильтруй датасет, оставив кандидатов с возрастом от `18` до `70` лет включительно.

In [16]:
print(df_candidates)

   candidate_id  age
0             1   22
1             2   25
2             3  190
3             4   30
4             5   28
5             6    5
6             7   35
7             8   42
8             9   29
9            10   31


In [17]:
print(f"Средний возраст: {df_candidates['age'].mean()}")  # Средний возраст с выбросами
print(f"Медианный возраст: {df_candidates['age'].median()}")  # Медианный возраст с выбросами
print(f"Медианный возраст: {df_candidates['age'].median()} оказался более устойчивым к выбросам")  # Выводим информацию о том, что медиана более устойчива к выбросам
print(f"Средний возраст (без выбросов): {df_candidates[df_candidates['age'].between(18, 70)]['age'].mean()}")  # Средний возраст без выбросов

Средний возраст: 43.7
Медианный возраст: 29.5
Медианный возраст: 29.5 оказался более устойчивым к выбросам
Средний возраст (без выбросов): 30.25


# Неделя 6: Статистика и корреляция

Раз мы разобрались с выбросами, давай сделаем первый шаг к предсказанию. Перед тем как строить модель, нужно понять: есть ли связь между признаками.

Для этого используется коэффициент корреляции Пирсона. Он показывает силу и направление линейной связи и принимает значения от `-1` до `1`.

- `1` — идеальная прямая связь.
- `-1` — идеальная обратная связь.
- `0` — связи нет.

## Задача
Представь, что ты исследуешь данные сотрудников IT-компании.

У тебя есть:
- стаж работы (`experience_years`),
- количество ошибок в коде за последний месяц (`bugs_count`).

Скопируй исходный датасет:

In [18]:
df_it = pd.DataFrame({
    'employee_id': [1, 2, 3, 4, 5, 6],
    'experience_years': [1, 2, 3, 5, 8, 10],      # Стаж работы
    'bugs_count': [25, 18, 12, 8, 3, 2]            # Количество багов в месяц
})

# Задание: Корреляция и выводы

Напиши код, который выполнит шаги:

## Шаг A
- Рассчитай матрицу корреляции между числовыми признаками с помощью `.corr()`.

## Шаг B
- Извлеки значение корреляции между `experience_years` и `bugs_count`.
- В комментарии опиши: сильная/слабая и прямая/обратная ли связь.

## Шаг C
- Представь джуна с `experience_years = 0` — ожидается ли у него больше багов или меньше на основании корреляции?

In [19]:
df_it

,employee_id,experience_years,bugs_count
0,1,1,25
1,2,2,18
2,3,3,12
3,4,5,8
4,5,8,3
5,6,10,2


In [20]:
print(df_it[['experience_years', 'bugs_count']].corr())
print(f'Обнаружена сильная обратная корреляция. Чем больше опыта - тем меньше ошибок.')
print(f'Ожидается, что сотрудник без опыта будет допускать больше ошибок, а с увеличением опыта количество ошибок будет снижаться.')


                  experience_years  bugs_count
experience_years          1.000000   -0.932223
bugs_count               -0.932223    1.000000
Обнаружена сильная обратная корреляция. Чем больше опыта - тем меньше ошибок.
Ожидается, что сотрудник без опыта будет допускать больше ошибок, а с увеличением опыта количество ошибок будет снижаться.


In [21]:
from scipy import stats

# 1. Считаем параметры линейной регрессии
# slope — наклон прямой, intercept — точка пересечения с осью Y (опыт = 0)
slope, intercept, r_value, p_value, std_err = stats.linregress(df_it['experience_years'], df_it['bugs_count'])

# 2. Делаем математический прогноз для experience_years = 0
target_experience = 0
predicted_bugs = slope * target_experience + intercept

print(f"Математически доказано:")
print(f"При опыте {target_experience} лет, ожидаемое количество багов составит: {predicted_bugs:.1f}")
print(f"С каждым годом опыта количество багов будет снижаться на: {abs(slope):.1f}")

Математически доказано:
При опыте 0 лет, ожидаемое количество багов составит: 22.7
С каждым годом опыта количество багов будет снижаться на: 2.4


# Неделя 7: Проверка гипотез и A/B тестирование

Переходим к одной из самых востребованных тем в Data Science — A/B-тестам.

Представь, что отдел дизайна сменил цвет кнопки «Купить» с синего на зеленый. Маркетологи утверждают, что новый цвет принес больше денег.

Чтобы это проверить, используют статистические тесты, например $t$-тест Стьюдента. Он помогает понять, является ли разница в средних чеках между группами статистически значимой.

### Что важно знать
- `p-value < 0.05` — разница статистически значима.
- `p-value >= 0.05` — разница может быть случайной.

## Твое задание
У тебя есть две выборки чеков: группа `A` (старый дизайн) и группа `B` (новый дизайн).

Скопируй исходный код и продолжи работу.

In [22]:
# Чек каждого пользователя в группах
group_A = [1200, 1500, 1100, 950, 1300, 1400, 1050, 1150, 1250, 1350]
group_B = [1400, 1650, 1500, 1200, 1380, 1600, 1450, 1300, 1550, 1700]

# Задание: A/B тестирование

Напиши код, который выполнит шаги:

## Шаг A
- Посчитай средний чек для группы `A` и группы `B`.

## Шаг B
- Проведи `t`-тест: `t_stat, p_value = stats.ttest_ind(group_A, group_B)`.
- Если `p_value < 0.05`, выведи: «Разница статистически значима, внедряем новый дизайн!»;
- Иначе выведи: «Разница случайна, оставляем все как было».

## Шаг C
- Покажи итоговый вывод на экран.

In [23]:
print(f'Средний чек в группе A: {np.mean(group_A):.2f}')
print(f'Средний чек в группе B: {np.mean(group_B):.2f}')
stats.ttest_ind(group_A, group_B)
t_stat, p_value = stats.ttest_ind(group_A, group_B)
print(f'T-статистика: {t_stat:.5f}, p-значение: {p_value:.5f}')
print(f"Разница статистически значима, внедряем новый дизайн!" if p_value < 0.05 else "Разница не статистически значима, оставляем старый дизайн.")

Средний чек в группе A: 1225.00
Средний чек в группе B: 1473.00
T-статистика: -3.39284, p-значение: 0.00324
Разница статистически значима, внедряем новый дизайн!


# Модуль 8-9: Косинусное сходство и ML-введение

Ты прошёл блок аналитики и статистики. Теперь мы открываем дверь в настоящее машинное обучение.

Любая модель, будь то линейная регрессия или нейросеть, видит данные как матрицы и векторы.

Представь, что у нас есть три пользователя стримингового сервиса. Их предпочтения по жанрам закодированы в векторах:

- Пользователь 1: `[5, 3, 0]` — любит экшен, средне комедии, не любит драмы.
- Пользователь 2: `[4, 3, 1]` — похожие вкусы.
- Пользователь 3: `[0, 1, 5]` — любит только драмы.

Чтобы рекомендовать фильмы, нам нужно измерить «сходство» между этими векторами. В Data Science часто используют косинусное сходство (`cosine similarity`).

Если угол между векторами равен 0 градусов (`cosine = 1`), то вкусы почти одинаковые.

## Задача
Мы не будем писать формулу вручную — воспользуемся `scikit-learn`.

Ниже — исходные данные.

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

# Векторы предпочтений трех пользователей (переводим в формат матриц NumPy)
user_1 = np.array([[5, 3, 0]])
user_2 = np.array([[4, 3, 1]])
user_3 = np.array([[0, 1, 5]])

# Задание: Косинусное сходство

Напиши код, который выполнит шаги:

## Шаг A
- Посчитай косинусное сходство между `user_1` и `user_2` с помощью `cosine_similarity(user_1, user_2)`.
- Выведи результат.

## Шаг B
- Посчитай косинусное сходство между `user_1` и `user_3`.
- Выведи результат.

## Шаг C
- Сравни два значения сходства и выведи на экран имя пользователя (`user_2` или `user_3`), вкусы которого ближе к `user_1`.

In [25]:
print(user_1, user_2, user_3)

[[5 3 0]] [[4 3 1]] [[0 1 5]]


In [26]:
print(f'косинусное сходство между user_1 и user_2: {cosine_similarity(user_1, user_2)}')  # Сравниваем пользователя 1 и 2)
print(f'косинусное сходство между user_1 и user_3: {cosine_similarity(user_1, user_3)}')  # Сравниваем пользователя 1 и 3)
print(f'Вкусы user_1 больше всего схожи с ' + ('user_2' if cosine_similarity(user_1, user_2) > cosine_similarity(user_1, user_3) else 'user_3'))  # Выводим, с кем больше всего схожи вкусы user_1   

косинусное сходство между user_1 и user_2: [[0.97537555]]
косинусное сходство между user_1 и user_3: [[0.10090092]]
Вкусы user_1 больше всего схожи с user_2


# Модуль 3: Классическое машинное обучение (Machine Learning)

Мы переходим к обучению полноценных моделей. Начнём с классики — линейной регрессии (`Linear Regression`).

Представь, что ты работаешь в агентстве недвижимости. У тебя есть данные о площади квартир и их стоимости. Задача модели — предсказывать цену по площади.

В машинном обучении данные обычно делят на две части:

- Обучающая выборка (`Train`): модель изучает зависимость.
- Тестовая выборка (`Test`): модель проверяется на новых данных.

## Задача
Мы будем использовать `scikit-learn` для обучения и оценки модели.

Ниже — исходные данные.

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Наш датасет: Площадь (X) и Цена (y)
data = {
    'area': [35, 45, 55, 60, 75, 80, 90, 110, 120, 150],
    'price': [3.5, 4.6, 5.4, 6.1, 7.6, 7.9, 9.2, 10.9, 12.1, 14.9]
}
df_housing = pd.DataFrame(data)

# Выделяем матрицу признаков X (всегда двумерная) и вектор целевой переменной y
X = df_housing[['area']]
y = df_housing['price']

# Задание: Линейная регрессия

Напиши код, который выполнит шаги:

## Шаг A
- Раздели данные на обучающую и тестовую выборки с помощью `train_test_split(X, y, test_size=0.2, random_state=42)`.
- На выходе должны получиться четыре переменные: `X_train`, `X_test`, `y_train`, `y_test`.

## Шаг B
- Создай модель `LinearRegression()` и обучи её на тренировочных данных методом `.fit()`.

## Шаг C
- Сделай предсказание для тестовой выборки: `y_pred = model.predict(X_test)`.
- Посчитай ошибку MAE через `mean_absolute_error(y_test, y_pred)`.
- Выведи значение MAE на экран.

In [28]:
train_test_split(X, y, test_size=0.2, random_state=42)

[   area
 5    80
 0    35
 7   110
 2    55
 9   150
 4    75
 3    60
 6    90,
    area
 8   120
 1    45,
 5     7.9
 0     3.5
 7    10.9
 2     5.4
 9    14.9
 4     7.6
 3     6.1
 6     9.2
 Name: price, dtype: float64,
 8    12.1
 1     4.6
 Name: price, dtype: float64]